# Spark Lakehouse Agent — Google Colab private MCP connectivity

This notebook joins a temporary Colab runtime to your Tailscale tailnet using userspace networking, then tests private HTTPS access and performs an MCP Streamable HTTP `initialize` handshake with Spark MCP and MinIO MCP on your OVH Ubuntu VM.

**Security:** use a dedicated ephemeral Tailscale auth key, stored in Colab Secrets as `TS_AUTHKEY`. Never put the key in notebook source, output, GitHub, or chat. The OVH MCP services must remain reachable through Tailscale Serve only; do not expose ports 8002/8003 publicly.

Before running: enable MagicDNS and HTTPS certificates in Tailscale; make sure your tailnet ACL/grants allow the Colab device to reach the OVH node on TCP 443. Replace the example hostname below with the exact hostname from `docker compose exec tailscale tailscale serve status`.

## 1. Configuration


In [ ]:
TAILNET_HOST = "spark-lakehouse-agent.<tailnet>.ts.net"  # Confirm with: docker compose exec tailscale tailscale serve status
SPARK_MCP_URL = f"https://{TAILNET_HOST}/mcp"
MINIO_MCP_URL = f"https://{TAILNET_HOST}:8443/mcp"
SOCKS_HOST = "127.0.0.1"
SOCKS_PORT = 1055
SOCKS_URL = f"socks5h://{SOCKS_HOST}:{SOCKS_PORT}"

print("Spark MCP:", SPARK_MCP_URL)
print("MinIO MCP:", MINIO_MCP_URL)
print("Proxy:", SOCKS_URL)

## 2. Create a dedicated ephemeral Tailscale auth key

Open the [Tailscale auth keys page](https://login.tailscale.com/admin/settings/keys). Create a short-lived **ephemeral** key for this disposable Colab node. Prefer a dedicated device tag such as `tag:colab`, and configure a least-privilege ACL/grant allowing that tag to reach only the OVH MCP node on TCP 443. Store the key in Colab's Secrets panel under the exact name `TS_AUTHKEY`; grant this notebook access. Do not print or hard-code it.

## 3. Install Tailscale and Python SOCKS support


In [ ]:
!curl -fsSL https://tailscale.com/install.sh | sh
!python -m pip -q install 'requests[socks]'

## 4. Start Tailscale userspace networking

Userspace mode does not require `/dev/net/tun` and exposes a local SOCKS5 proxy. Applications must explicitly use that proxy. The `mem:` state store is appropriate for a disposable Colab runtime.

In [ ]:
import os
import subprocess
import time
from google.colab import userdata

auth_key = userdata.get("TS_AUTHKEY")
if not auth_key:
    raise RuntimeError("Missing TS_AUTHKEY. Add it in Colab Secrets and enable notebook access.")

tailscaled = subprocess.Popen(
    [
        "tailscaled",
        "--state=mem:",
        "--tun=userspace-networking",
        f"--socks5-server={SOCKS_HOST}:{SOCKS_PORT}",
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
)

# Wait for the local daemon control socket to become ready.
daemon_ready = False
for _ in range(30):
    probe = subprocess.run(["tailscale", "status"], capture_output=True, text=True)
    if probe.returncode == 0:
        daemon_ready = True
        break
    time.sleep(1)
if not daemon_ready:
    raise RuntimeError("tailscaled did not become ready; inspect the Colab runtime and rerun this setup cell.")

try:
    subprocess.run(
        ["tailscale", "up", f"--auth-key={auth_key}", "--hostname=colab-mcp"],
        check=True,
        text=True,
        timeout=90,
    )
finally:
    # Drop the Python reference to the secret after enrollment.
    del auth_key

print(subprocess.run(["tailscale", "status"], check=True, capture_output=True, text=True).stdout)
print("Tailscale userspace node is connected; SOCKS5 proxy:", SOCKS_URL)

## 5. Test private DNS, TCP reachability and TLS through SOCKS5

`socks5h` makes the hostname resolution happen through the proxy path. Do not use `socks5` for this test because that can cause local DNS resolution outside the tailnet. A plain HTTP GET to an MCP endpoint is only a transport smoke test; HTTP 400/405 can be normal because GET is not the MCP initialize handshake.

In [ ]:
import requests
from requests.exceptions import RequestException

proxies = {"http": SOCKS_URL, "https": SOCKS_URL}
for label, url in [("Spark MCP", SPARK_MCP_URL), ("MinIO MCP", MINIO_MCP_URL)]:
    try:
        response = requests.get(
            url,
            headers={"Accept": "text/event-stream"},
            proxies=proxies,
            stream=True,
            timeout=(10, 5),
        )
        print(f"{label}: HTTP {response.status_code}; TLS verified (request completed)")
        print("  Content-Type:", response.headers.get("content-type", "(not supplied)"))
        response.close()
        if response.status_code in (401, 403):
            print("  Network path works, but the service or an access policy denied this request.")
    except RequestException as exc:
        print(f"{label}: CONNECTION FAILED — {type(exc).__name__}: {exc}")
        raise

## 6. Perform an MCP Streamable HTTP initialize handshake

This sends the JSON-RPC `initialize` request and then the `notifications/initialized` notification. It verifies more than a browser GET. The request uses the commonly supported protocol version `2025-03-26`; if the server reports that version as unsupported, inspect the server's supported MCP version and update `MCP_PROTOCOL_VERSION` accordingly. TLS certificate verification remains enabled.

In [ ]:
import json
import uuid

MCP_PROTOCOL_VERSION = "2025-03-26"
MCP_HEADERS = {
    "Content-Type": "application/json",
    "Accept": "application/json, text/event-stream",
}

def _jsonrpc_payload(response):
    """Decode either an application/json response or the first SSE data event."""
    content_type = response.headers.get("content-type", "").lower()
    if "application/json" in content_type:
        return response.json()
    if "text/event-stream" in content_type:
        for line in response.text.splitlines():
            if line.startswith("data:"):
                return json.loads(line[len("data:"):].strip())
    raise RuntimeError(
        f"Unexpected MCP initialize response: HTTP {response.status_code}, "
        f"Content-Type={content_type!r}, body={response.text[:1000]!r}"
    )

def mcp_initialize(label, url):
    request_id = str(uuid.uuid4())
    initialize_request = {
        "jsonrpc": "2.0",
        "id": request_id,
        "method": "initialize",
        "params": {
            "protocolVersion": MCP_PROTOCOL_VERSION,
            "capabilities": {},
            "clientInfo": {"name": "spark-lakehouse-colab-validator", "version": "1.0.0"},
        },
    }
    response = requests.post(
        url,
        json=initialize_request,
        headers=MCP_HEADERS,
        proxies=proxies,
        timeout=30,
    )
    if response.status_code not in (200, 201):
        raise RuntimeError(f"{label} initialize failed: HTTP {response.status_code}: {response.text[:1500]}")
    result = _jsonrpc_payload(response)
    if "error" in result:
        raise RuntimeError(f"{label} MCP initialize error: {result['error']}")
    if "result" not in result:
        raise RuntimeError(f"{label} returned no JSON-RPC result: {result}")

    session_id = response.headers.get("Mcp-Session-Id")
    notification_headers = dict(MCP_HEADERS)
    if session_id:
        notification_headers["Mcp-Session-Id"] = session_id
    initialized_notification = {
        "jsonrpc": "2.0",
        "method": "notifications/initialized",
    }
    notification_response = requests.post(
        url,
        json=initialized_notification,
        headers=notification_headers,
        proxies=proxies,
        timeout=30,
    )
    if notification_response.status_code not in (200, 202, 204):
        raise RuntimeError(
            f"{label} initialized notification failed: HTTP "
            f"{notification_response.status_code}: {notification_response.text[:1000]}"
        )

    server_info = result["result"].get("serverInfo", {})
    negotiated_version = result["result"].get("protocolVersion", "not reported")
    print(f"PASS — {label}: MCP initialize accepted")
    print("  Server:", server_info.get("name", "unknown"), server_info.get("version", ""))
    print("  Protocol:", negotiated_version)
    print("  Session-based:", bool(session_id))
    print("  Initialized notification: HTTP", notification_response.status_code)
    return result["result"]

spark_initialize_result = mcp_initialize("Spark MCP", SPARK_MCP_URL)
minio_initialize_result = mcp_initialize("MinIO MCP", MINIO_MCP_URL)

## 7. Results and interpretation

A successful run means the Colab runtime joined the tailnet, reached both HTTPS endpoints through the SOCKS5 proxy, passed TLS verification, and completed the MCP initialize exchange with both services. It does not yet validate individual tool behavior, Spark query permissions, MinIO bucket policy, or write operations. Test tools separately with read-only operations first.

In [ ]:
print("COLAB PRIVATE MCP CONNECTIVITY SUMMARY")
print("Tailnet hostname:", TAILNET_HOST)
print("Spark MCP initialized:", bool(spark_initialize_result))
print("MinIO MCP initialized:", bool(minio_initialize_result))
print("Proxy:", SOCKS_URL)
print("Keep MCP writes disabled until tool-level authorization and data policies are validated.")

## 8. Stop the temporary Colab Tailscale daemon

Run when finished. The auth key is ephemeral and the runtime is disposable; the node should be removed from the tailnet after it goes offline.

In [ ]:
tailscaled.terminate()
try:
    tailscaled.wait(timeout=10)
except subprocess.TimeoutExpired:
    tailscaled.kill()
    tailscaled.wait(timeout=5)
print("Stopped the Colab Tailscale userspace daemon.")

## Troubleshooting

| Symptom | Likely cause / action |
|---|---|
| Auth key rejected | Check expiry, ephemeral setting, tag authorization and that the key has not been revoked. |
| Colab cannot reach SOCKS proxy | Confirm `tailscaled` is still running and port 1055 is listening. Rerun the setup cell only after cleaning up an old daemon. |
| DNS error through proxy | Confirm the exact hostname from `docker compose exec tailscale tailscale serve status`; use `socks5h`, not `socks5`. |
| TCP timeout | Check tailnet membership and ACL/grant rules for the Colab tag to the OVH node on TCP 443/8443. |
| TLS certificate error | Enable Tailscale HTTPS certificates and inspect `docker compose exec tailscale tailscale serve status` on OVH. Do not disable TLS verification. |
| HTTP 400/405 on GET | Not necessarily a fault; use the initialize cell to test MCP. |
| MCP initialize rejected | Check protocol version compatibility, Streamable HTTP transport, service logs and Serve routing. |

On the OVH VM, useful diagnostics are `tailscale status`, `docker compose exec tailscale tailscale serve status`, `docker compose ps spark-mcp minio-mcp`, and `docker compose logs --tail=100 spark-mcp minio-mcp`.

Security reminders: do not enable Tailscale Funnel; do not expose Docker MCP ports 8002/8003 publicly; keep host bindings on `127.0.0.1`; restrict tailnet ACLs; use least-privilege MinIO credentials; never publish the Colab auth key.